# Deepfake (CIFAKE) detector — training v3 (fast)

Same idea as v2 (88.7% test), but much faster:
- **Duplicates removed automatically** — each image is used once (your folder had ~2× the official CIFAKE count).
- **UP_SIZE = 128** instead of 224 → ~3× less compute per image.
- **Phase 1 = 1 epoch, Phase 2 = 5 epochs** with a schedule that actually finishes (v2 was set for 10 and stopped at 2).
- **Safe saving** — training writes to `model_v3.keras`; `model.keras` (the app's model) is only replaced if the new test accuracy is higher.

Just run all cells from top to bottom.

In [1]:
import os, json, hashlib, random
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

ROOT = Path.cwd()
while not (ROOT / "backend" / "train").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR  = ROOT / "backend" / "train" / "dataset"
MODEL_DIR = ROOT / "backend" / "saved_models"
print("Data:", DATA_DIR)
assert (DATA_DIR / "train").exists() and (DATA_DIR / "test").exists(), f"dataset not found at {DATA_DIR}"

# ---- settings ----
IMG_SIZE  = 32       # native CIFAKE size; the backend resizes uploads to this
UP_SIZE   = 128      # what EfficientNet sees (224 = slower, slightly better)
BATCH     = 128
EPOCHS_P1 = 1
EPOCHS_P2 = 5
SEED      = 42
AUTOTUNE  = tf.data.AUTOTUNE

TF: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Data: /Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/train/dataset


## 1. Build a clean, duplicate-free file list
Every image is hashed. Exact duplicates are used only once, and any training image that also appears in the test set is dropped from training (so the test score is honest). Takes a minute or two.

In [2]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def class_dirs(split):
    dirs = sorted(d for d in (DATA_DIR / split).iterdir() if d.is_dir())
    names = [d.name.lower() for d in dirs]
    assert names == ["fake", "real"], f"expected fake/real folders in {split}, found {names}"
    return dirs                       # index 0 = FAKE, 1 = REAL

def unique_files(split, exclude=frozenset()):
    seen, paths, labels, n_total = set(), [], [], 0
    for label, d in enumerate(class_dirs(split)):
        for p in d.rglob("*"):
            if p.suffix.lower() not in IMG_EXT:
                continue
            n_total += 1
            h = hashlib.md5(p.read_bytes()).hexdigest()
            if h in seen or h in exclude:
                continue
            seen.add(h); paths.append(str(p)); labels.append(label)
    labels = np.array(labels, dtype=np.float32)
    print(f"{split:5s}: {n_total:>7,d} files -> {len(paths):>7,d} unique used "
          f"(FAKE {int((labels == 0).sum()):,}, REAL {int((labels == 1).sum()):,})")
    return paths, labels, seen

test_paths, test_labels, test_hashes = unique_files("test")
all_paths, all_labels, _ = unique_files("train", exclude=test_hashes)

# 90/10 train/validation split
idx = np.random.default_rng(SEED).permutation(len(all_paths))
n_val = len(idx) // 10
val_idx, tr_idx = idx[:n_val], idx[n_val:]
tr_paths,  tr_labels  = [all_paths[i] for i in tr_idx],  all_labels[tr_idx]
val_paths, val_labels = [all_paths[i] for i in val_idx], all_labels[val_idx]
print(f"train {len(tr_paths):,} | val {len(val_paths):,} | test {len(test_paths):,}")

test :  30,000 files ->  30,000 unique used (FAKE 10,000, REAL 20,000)
train: 200,000 files -> 199,080 unique used (FAKE 99,080, REAL 100,000)
train 179,172 | val 19,908 | test 30,000


## 2. Data pipeline

In [3]:
def load_img(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))          # float32, values 0-255
    return img, tf.expand_dims(label, -1)

def make_ds(paths, labels, train=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE).cache()
    if train:
        ds = ds.shuffle(20_000, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(tr_paths, tr_labels, train=True)
val_ds   = make_ds(val_paths, val_labels)
test_ds  = make_ds(test_paths, test_labels)

2026-11-14 15:49:21.286346: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-11-14 15:49:21.286371: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-11-14 15:49:21.286434: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-11-14 15:49:21.286448: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-11-14 15:49:21.286461: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## 3. Model (input 32×32 → upsample to UP_SIZE inside the model)

In [4]:
base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(UP_SIZE, UP_SIZE, 3))
base.trainable = False

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = layers.RandomFlip("horizontal")(inputs)
x = layers.Resizing(UP_SIZE, UP_SIZE, interpolation="bilinear")(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)      # P(REAL)
model = tf.keras.Model(inputs, outputs)
METRICS = ["accuracy", tf.keras.metrics.AUC(name="auc")]

## 4. Phase 1 — head only (1 epoch)

In [5]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P1)

/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-11-14 15:49:24.292624: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1400/1400 ━━━━━━━━━━━━━━━━━━━━ 463s 324ms/step - accuracy: 0.7514 - auc: 0.8353 - loss: 0.4991 - val_accuracy: 0.7888 - val_auc: 0.8767 - val_loss: 0.4436


## 5. Phase 2 — fine-tune the whole backbone (5 epochs)
Learning rate starts at 3e-4 and decays to almost zero by the last epoch. Let it run to the end — the last epochs, at low learning rate, are where the final accuracy comes from. The best epoch is saved to `model_v3.keras`.

In [6]:
base.trainable = True
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

steps = int(train_ds.cardinality())
lr = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=3e-4, decay_steps=steps * EPOCHS_P2, alpha=0.01)
model.compile(optimizer=tf.keras.optimizers.AdamW(lr, weight_decay=1e-4),
              loss="binary_crossentropy", metrics=METRICS)

V3_PATH = MODEL_DIR / "model_v3.keras"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P2,
          callbacks=[tf.keras.callbacks.ModelCheckpoint(
              str(V3_PATH), monitor="val_accuracy", mode="max", save_best_only=True)])

model = tf.keras.models.load_model(V3_PATH, compile=False)   # best epoch

Epoch 1/5
1400/1400 ━━━━━━━━━━━━━━━━━━━━ 1634s 1s/step - accuracy: 0.8805 - auc: 0.9491 - loss: 0.2695 - val_accuracy: 0.9060 - val_auc: 0.9778 - val_loss: 0.2180
Epoch 2/5
1400/1400 ━━━━━━━━━━━━━━━━━━━━ 2652s 2s/step - accuracy: 0.9232 - auc: 0.9788 - loss: 0.1859 - val_accuracy: 0.9297 - val_auc: 0.9831 - val_loss: 0.1706
Epoch 3/5
1400/1400 ━━━━━━━━━━━━━━━━━━━━ 1387s 990ms/step - accuracy: 0.9438 - auc: 0.9878 - loss: 0.1400 - val_accuracy: 0.9407 - val_auc: 0.9894 - val_loss: 0.1462
Epoch 4/5
 239/1400 ━━━━━━━━━━━━━━━━━━━━ 19:00 982ms/step - accuracy: 0.9528 - auc: 0.9914 - loss: 0.1169

KeyboardInterrupt: 

In [7]:
model = tf.keras.models.load_model(V3_PATH, compile=False)

## 6. Test-set evaluation

In [8]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, roc_auc_score

y_true = test_labels.astype(int)
p_real = model.predict(test_ds, verbose=0).ravel()
p_flip = model.predict(test_ds.map(lambda x, y: (tf.image.flip_left_right(x), y)), verbose=0).ravel()
p_tta  = (p_real + p_flip) / 2

for name, p in [("plain", p_real), ("flip-TTA", p_tta)]:
    y_pred = (p > 0.5).astype(int)
    print(f"{name:8s} acc={np.mean(y_pred == y_true):.4f}  "
          f"balanced_acc={balanced_accuracy_score(y_true, y_pred):.4f}  AUC={roc_auc_score(y_true, p):.4f}")

y_pred = (p_real > 0.5).astype(int)
print(classification_report(y_true, y_pred, target_names=["FAKE", "REAL"], digits=4))
print(confusion_matrix(y_true, y_pred))
test_acc = float(np.mean(y_pred == y_true))

plain    acc=0.9247  balanced_acc=0.9330  AUC=0.9838
flip-TTA acc=0.9317  balanced_acc=0.9400  AUC=0.9867
              precision    recall  f1-score   support

        FAKE     0.8390    0.9580    0.8946     10000
        REAL     0.9774    0.9081    0.9415     20000

    accuracy                         0.9247     30000
   macro avg     0.9082    0.9330    0.9180     30000
weighted avg     0.9313    0.9247    0.9258     30000

[[ 9580   420]
 [ 1838 18162]]


## 7. Save — only if better than the current app model
Note: v3 tests on the de-duplicated test set, so its number is the more honest one; it can differ slightly from v2's 88.7% for that reason alone.

In [9]:
old_acc = json.load(open(MODEL_DIR / "meta.json")).get("test_accuracy", 0) if (MODEL_DIR / "meta.json").exists() else 0
print(f"current app model: {old_acc:.4f} | new v3 model: {test_acc:.4f}")

if test_acc > old_acc:
    model.save(MODEL_DIR / "model.keras")
    meta = {
        "model": "EfficientNetB0",
        "input_size": [IMG_SIZE, IMG_SIZE],
        "backbone_input_size": [UP_SIZE, UP_SIZE],
        "labels": {"0": "FAKE", "1": "REAL"},
        "preprocessing": "RGB, resize to 32x32, keep values 0-255 (no rescaling); model upsamples internally",
        "output": "sigmoid, P(REAL)",
        "test_accuracy": test_acc,
    }
    json.dump(meta, open(MODEL_DIR / "meta.json", "w"), indent=2)
    print("Saved: the app now uses the v3 model. Restart the backend.")
else:
    print("Not saved: the current model.keras is kept.")

current app model: 0.8872 | new v3 model: 0.9247
Saved: the app now uses the v3 model. Restart the backend.
